
# 21. ADAMACS Schema Dependency Diagrams (Read-Only)

This notebook renders **one schema at a time** using `dj.Diagram` and saves SVG files.

Safety:
- read-only: no `insert`, `populate`, `delete`, or `drop`
- intended for documentation, onboarding, and dependency debugging

Output files:
- `notebooks/schema_diagrams/*.svg`


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=True)
repo_root = ctx.repo_root

import datajoint as dj

from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
if not ctx.dot_path:
    raise RuntimeError(
        "Graphviz 'dot' executable is not available on PATH. Install graphviz in this environment first."
    )


In [ ]:
# Import pipeline-linked schemas using the configured ADAMACS linking module.
from adamacs import pipeline as pl
from adamacs.schemas import (
    subject,
    surgery,
    equipment,
    behavior,
    mocap,
    pupil_tracking,
    virtual_markers_optitrack,
    disk,
)

try:
    from adamacs.schemas import denoising
except Exception as exc:
    denoising = None
    print(f"Skipping optional schema 'denoising': {exc}")

session = pl.session
event = pl.event
trial = pl.trial
scan = pl.scan
imaging = pl.imaging
model = pl.model

schema_registry = {
    "subject": subject.schema,
    "surgery": surgery.schema,
    "equipment": equipment.schema,
    "behavior": behavior.schema,
    "mocap": mocap.schema,
    "pupil_tracking": pupil_tracking.schema,
    "virtual_markers_optitrack": virtual_markers_optitrack.schema,
    "disk": disk.schema,
    "session": session.schema,
    "event": event.schema,
    "trial": trial.schema,
    "scan": scan.schema,
    "imaging": imaging.schema,
    "model": model.schema,
}

if denoising is not None:
    schema_registry["denoising"] = denoising.schema



## Schema explanations (parent-key view)

Each schema is documented with its typical parent key space and role in the end-to-end pipeline.


In [ ]:
schema_definitions = [
    {
        "name": "subject",
        "parent": "subject",
        "description": "Animal-level metadata backbone. Defines stable subject identity, lab ownership, protocol context, and user/project linkage used by downstream session and recording tables.",
    },
    {
        "name": "surgery",
        "parent": "subject",
        "description": "Implant and anatomical intervention history. Stores coordinate/site/procedure details that contextualize recording location and hardware constraints.",
    },
    {
        "name": "equipment",
        "parent": "equipment/device",
        "description": "Acquisition hardware and rig definitions. Normalizes scanner/camera/device metadata so ingest tasks can resolve hardware-specific parsing and processing parameters.",
    },
    {
        "name": "session",
        "parent": "subject + session_id",
        "description": "Session-level administrative and directory metadata. Connects subject identity to concrete acquisition sessions and links user ownership and storage paths.",
    },
    {
        "name": "scan",
        "parent": "session + scan_id",
        "description": "Imaging scan-level ingestion metadata. Encodes scan paths, field information, imaging setup metadata, and identifiers used as central join keys across modalities.",
    },
    {
        "name": "imaging",
        "parent": "scan",
        "description": "Calcium imaging processing outputs. Covers processing tasks, curation states, segmentation, fluorescence traces, and activity extraction products.",
    },
    {
        "name": "model",
        "parent": "scan/session + recording_id",
        "description": "DLC/model-based video processing stack. Tracks video recordings, model definitions, pose-estimation tasks, and inferred body-part trajectories.",
    },
    {
        "name": "event",
        "parent": "session/scan time axis",
        "description": "Event timeline integration layer. Stores behavior recording references and event timestamps used to align neural, behavioral, and camera streams.",
    },
    {
        "name": "trial",
        "parent": "session/scan + trial_id",
        "description": "Trialization and behavioral epoch structure. Defines trial boundaries and trial-event anchors for task-level and latency analyses.",
    },
    {
        "name": "behavior",
        "parent": "scan/session",
        "description": "Behavioral stream ingestion outputs. Includes harp/treadmill/camera-sync channels and synchronized behavior-side continuous data products.",
    },
    {
        "name": "mocap",
        "parent": "scan/session",
        "description": "Raw motion-capture integration. Ingests and structures OptiTrack trajectories and timing needed for rigid-body and gaze reconstruction.",
    },
    {
        "name": "virtual_markers_optitrack",
        "parent": "mocap + scan",
        "description": "Derived rigid-body reconstruction layer. Builds virtual marker/head pose estimates from mocap inputs to support eye/head/world alignment.",
    },
    {
        "name": "pupil_tracking",
        "parent": "scan + eye recording",
        "description": "Eye-camera and gaze-reconstruction outputs. Fits pupil ellipses, infers 3D gaze vectors, and links eye measurements with rigid-body orientation.",
    },
    {
        "name": "denoising",
        "parent": "imaging processing output",
        "description": "Post-processing denoising tasks for imaging outputs. Captures denoising parameterization and generated denoised signal products.",
    },
    {
        "name": "disk",
        "parent": "scan/session storage",
        "description": "Storage/indexing helpers for large artifacts. Tracks disk-side dataset products and intermediate outputs needed by ingest and post-processing routines.",
    },
]

schema_specs = [
    {
        "name": spec["name"],
        "parent": spec["parent"],
        "schema": schema_registry[spec["name"]],
        "description": spec["description"],
    }
    for spec in schema_definitions
    if spec["name"] in schema_registry
]

pd.DataFrame(
    [
        {
            "schema": s["name"],
            "parent_key_root": s["parent"],
            "description": s["description"],
        }
        for s in schema_specs
    ]
)



In [ ]:

output_dir = repo_root / "notebooks" / "schema_diagrams"
output_dir.mkdir(parents=True, exist_ok=True)

warning_rows = []
render_rows = []

for spec in schema_specs:
    display(Markdown(f"## {spec['name']} schema"))
    display(Markdown(f"**Parent key root:** `{spec['parent']}`"))
    display(Markdown(spec["description"]))

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        diagram = dj.Diagram(spec["schema"]) + 1 - 1
        display(diagram)

        out_svg = output_dir / f"{spec['name']}.svg"
        diagram.save(str(out_svg), format="svg")

    render_rows.append(
        {
            "schema": spec["name"],
            "svg_path": str(out_svg),
            "nodes_shown": len(diagram.nodes_to_show),
            "warning_count": len(caught),
        }
    )

    for item in caught:
        warning_rows.append(
            {
                "schema": spec["name"],
                "warning_type": item.category.__name__,
                "warning_message": str(item.message),
            }
        )

print(f"Saved schema diagrams to: {output_dir}")
render_df = pd.DataFrame(render_rows)
render_df


In [ ]:

warning_df = pd.DataFrame(warning_rows)
if warning_df.empty:
    print("No warnings captured while rendering dj.Diagram outputs.")
else:
    warning_df
